**To do:**
* particle interactions
* materials definition
* insert particles : using box or otherwise
* param  sweep - what do we look at - e.g. friction

# Uniaxial Compression Testing 

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tempfile

import ansys.rocky.core as pyrocky

## Setup

### Run-based variables

In [2]:
particle_box_length = 10 / 1000 # 10 mm
particle_radius = 150e-6 # 150 microns
particle_density = 2500 # kg/m^3

### Define setup directory

In [3]:
project_dir ="/home/rocky-vm/Documents"
# project_dir = tempfile.mkdtemp(prefix='rocky_',)  #UNCOMMENT FOR A TEMPORARY PROJECT

### Launch / connect to rocky + open project

In [5]:
# try:
#     print("Connecting to Rocky...")
#     rocky = pyrocky.connect_to_rocky()
# except Exception as e:
#     print("Rocky is not running. Starting Rocky...")
#     rocky = pyrocky.launch_rocky(rocky_exe='/home/rocky-vm/ansys_inc/v242/rocky/bin/Rocky', headless=False) #UNCOMMENT WHEN INSTANTIATING ROCKY 

rocky = pyrocky.launch_rocky(rocky_exe='/home/rocky-vm/ansys_inc/v242/rocky/bin/Rocky', headless=False)
project = rocky.api.CreateProject()
project.SaveProject(os.path.join(project_dir, "compression-testing.rocky"))
study = project.GetStudy()

Installed Qt WebEngine locales directory not found at location /home/rocky-vm/ansys_inc/v242/rocky/bin/lib/PyQt5/translations/qtwebengine_locales. Trying application directory...
Qt WebEngine locales directory not found at location /home/rocky-vm/ansys_inc/v242/rocky/bin/qtwebengine_locales. Trying fallback directory... Translations MAY NOT not be correct.
[0328/011609.293292:WARNING:resource_bundle_qt.cpp(116)] locale_file_path.empty() for locale 


## Parameters Setup

### Loading  meshes

#### Compression Meshes

In [6]:
# Get mesh directories
# Only absolute paths work!
mesh_dir = os.path.abspath('../meshdir/compression/')
mesh1_name = 'square_wall_negZ.stl'
mesh2_name = 'square_wall_posZ.stl'

os.path.join(mesh_dir, mesh1_name)


compr_wall1= study.ImportWall(os.path.join(mesh_dir, mesh1_name), import_scale=1.0, convert_yz=True) [0]
compr_wall2 = study.ImportWall(os.path.join(mesh_dir, mesh2_name), import_scale=1.0, convert_yz=True)[0]
compr_wall1.SetName('compressing_wall1')
compr_wall1.SetName('compressing_wall2')

# Set the walls to be 1 m apart at start
compr_wall1.SetTranslation([-0.5, 0, 0])
compr_wall2.SetTranslation([ 0.5, 0, 0])

#### Insertion Support Meshes

In [ ]:
insert_base_name = 'insert_base.stl'
insert_support3_name = 'insert_support1.stl'
insert_support4_name = 'insert_support2.stl'
insert_inlet_name = 'insert_inlet.stl'

shift_magnitude = 1 - particle_box_length
# Default positions are for a 1 m box - changed to be equivalent to box size

insert_support1= study.ImportWall(os.path.join(mesh_dir, mesh1_name), import_scale=particle_box_length, convert_yz=True) [0]
insert_support1.SetName('insert_support1')
insert_support1.SetTranslation([-particle_box_length/2, 0, 0])

insert_support2 = study.ImportWall(os.path.join(mesh_dir, mesh2_name), import_scale=particle_box_length, convert_yz=True)[0]
insert_support2.SetName('insert_support2')
insert_support2.SetTranslation([particle_box_length/2, 0, 0])

insert_base = study.ImportWall(os.path.join(mesh_dir, insert_base_name), import_scale=particle_box_length, convert_yz=False)[0]
insert_base.SetName('insert_base')
insert_base.SetTranslation([0, 0, 0])


insert_support3 = study.ImportWall(os.path.join(mesh_dir, insert_support3_name), import_scale=particle_box_length, convert_yz=False)[0]
insert_support3.SetName('insert_support3')
insert_support3.SetTranslation([0, 0, 0])

insert_support4 = study.ImportWall(os.path.join(mesh_dir, insert_support4_name), import_scale=particle_box_length, convert_yz=False)[0]
insert_support4.SetName('insert_support4')
insert_support4.SetTranslation([0, 0, 0])

insert_inlet = study.ImportSurface(os.path.join(mesh_dir, insert_inlet_name), import_scale=particle_box_length, convert_yz=False)[0]
insert_inlet.SetName('insert_inlet')
insert_inlet.SetTranslation([0, 0, 0])


### Define Materials

In [ ]:
material_collection = study.GetMaterialCollection()
material_collection.Clear()

particle_mat = material_collection.AddSolidMaterial()
particle_mat.SetName("Particle Material")
particle_mat.SetDensity(2700, 'kg/m3')
particle_mat.SetYoungsModulus(5e6, 'Pa')
particle_mat.SetPoissonRatio(0.3)
particle_mat.SetUseBulkDensity(False)

wall_mat = material_collection.AddSolidMaterial()
wall_mat.SetName("Wall Material")
wall_mat.SetDensity(2700, 'kg/m3')
wall_mat.SetYoungsModulus(5e6, 'Pa')
wall_mat.SetPoissonRatio(0.3)
wall_mat.SetUseBulkDensity(False)

### Define particle params

In [ ]:
particle = study.CreateParticle() # creates a 'default' particle
size_distr_list = particle.GetSizeDistributionList()
size_distr_list.Clear() # clear the default size distribution

psd = size_distr_list.New()
psd.SetSize(150e-6,'m')
psd.SetCumulativePercentage(100)

### Define Domain settings

In [ ]:
domain=  study.GetDomainSettings()
domain.SetDomainType('CARTESIAN')
domain.SetUseBoundaryLimits(True)
domain.SetCartesianPeriodicDirections('XY') #no periodic in Z-direction to allow compression
domain.SetPeriodicLimitsMaxCoordinates([particle_box_size,particle_box_size,particle_box_size])
domain.SetPeriodicLimitsMinCoordinates([-particle_box_size,-particle_box_size,-particle_box_size])

### Insertion Settings

### Interaction Settings

In [ ]:
interaction_collection = study.GetMaterialsInteractionCollection()

pw_interaction = interaction_collection.GetMaterialsInteraction(particle_mat, wall_mat)
pp_interaction = interaction_collection.GetMaterialsInteraction(particle_mat, particle_mat)
ww_interaction = interaction_collection.GetMaterialsInteraction(wall_mat, wall_mat)


## Close Rocky

In [ ]:
rocky.close()